In [ ]:
#
# This is a sample Notebook to demonstrate how to read "MNIST Dataset"
#
import numpy as np # linear algebra
import struct
from array import array
from os.path  import join

#
# MNIST Data Loader Class
#
class MnistDataloader(object):
    def __init__(self, training_images_filepath,training_labels_filepath,
                 test_images_filepath, test_labels_filepath):
        self.training_images_filepath = training_images_filepath
        self.training_labels_filepath = training_labels_filepath
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath
    
    def read_images_labels(self, images_filepath, labels_filepath):        
        labels = []
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError('Magic number mismatch, expected 2049, got {}'.format(magic))
            labels = array("B", file.read())        
        
        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError('Magic number mismatch, expected 2051, got {}'.format(magic))
            image_data = array("B", file.read())        
        images = []
        flattened_images = []
        for i in range(size):
            images.append([0] * rows * cols)
            flattened_images.append([0]*rows*cols)
        for i in range(size):
            img = np.array(image_data[i * rows * cols:(i + 1) * rows * cols])
            flt_img = img.copy()
            img = img.reshape(28, 28)
            flattened_images[i]            
            images[i][:] = img            
            flattened_images[i][:] = flt_img
            
        return flattened_images, images, labels
            
    def load_data(self):
        x_train_flat, x_train, y_train = self.read_images_labels(self.training_images_filepath, self.training_labels_filepath)
        x_test_flat, x_test, y_test = self.read_images_labels(self.test_images_filepath, self.test_labels_filepath)
        return (x_train_flat, x_train, y_train),(x_test_flat, x_test, y_test)        

#
# Verify Reading Dataset via MnistDataloader class
#
import random
import matplotlib.pyplot as plt

#
# Set file paths based on added MNIST Datasets
#
input_path = './archive'
training_images_filepath = join(input_path, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
training_labels_filepath = join(input_path, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
test_images_filepath = join(input_path, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
test_labels_filepath = join(input_path, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

#
# Helper function to show a list of images with their relating titles
#
def show_images(images, title_texts):
    cols = 5
    rows = int(len(images)/cols) + 1
    plt.figure(figsize=(30,20))
    index = 1    
    for x in zip(images, title_texts):        
        image = x[0]        
        title_text = x[1]
        plt.subplot(rows, cols, index)        
        plt.imshow(image, cmap=plt.cm.gray)
        if (title_text != ''):
            plt.title(title_text, fontsize = 15);        
        index += 1

#
# Load MINST dataset
#
mnist_dataloader = MnistDataloader(training_images_filepath, training_labels_filepath, test_images_filepath, test_labels_filepath)
(x_train_flat, x_train, y_train), (x_test_flat, x_test, y_test) = mnist_dataloader.load_data()

#
# Show some random training and test images 
#
images_2_show = []
titles_2_show = []
for i in range(0, 10):
    r = random.randint(1, 60000)
    images_2_show.append(x_train[r])
    titles_2_show.append('training image [' + str(r) + '] = ' + str(y_train[r]))    

for i in range(0, 5):
    r = random.randint(1, 10000)
    images_2_show.append(x_test[r])        
    titles_2_show.append('test image [' + str(r) + '] = ' + str(y_test[r]))    

show_images(images_2_show, titles_2_show)


np_x_train_flat = np.array(x_train_flat)
int_y_train = np.array([int(i) for i in y_train])

np_x_test_flat = np.array(x_test_flat)
int_y_test = np.array([int(i) for i in y_test])

In [ ]:
import math
import numpy as np

activations = []
weighted_sums = []
beta_1 = 0.90
beta_2 = 0.999
learning_rate = 0.001
moment_1_history = {}
moment_2_history = {}

bias_moment_1_history = {}
bias_moment_2_history = {}

epsilon = 1e-8

In [ ]:
def sparse_categorical_crossentropy(batch_predictions, true_classes):
    preds = np.asarray(batch_predictions)
    targets = np.asarray(true_classes)
    eps = 1e-15
    preds = np.clip(preds, eps, 1-eps)
    true_probabilities = preds[np.arange(len(targets)),targets]
    return -np.mean(np.log(true_probabilities))

In [ ]:
def softmax(output_layer_output):
    max_per_sample = np.array([[max(sample_output) for i in range(len(sample_output))] for sample_output in output_layer_output])
    new_output_layer = output_layer_output - max_per_sample
    numerator = np.exp(new_output_layer)
    
    result = numerator/np.sum(numerator,axis=1, keepdims=True)
    
    return result


In [ ]:
def accuracy(batch_predictions, true_classes):
    preds = np.asarray(batch_predictions)
    targets = np.asarray(true_classes)
    
    predicted_classes = np.argmax(preds, axis=1)
    matches = (predicted_classes ==  targets)
    accuracy = np.mean(matches)

    print(f"Predicted: {predicted_classes}")
    print(f"Matches:   {matches}")
    print(f"Accuracy:  {accuracy:.2%}")
    

In [ ]:
def softmax_sparse_categorical_crossentropy_gradient(logits_prediction_arr, true_class_idx):
    grad = logits_prediction_arr.copy()
    for i in range(len(logits_prediction_arr)):
        grad[i][true_class_idx[i]]-=1
    return grad

In [ ]:
def hidden_layer_backprop_gradients_for_layer(curr_pre_activations, next_layers_error_gradients, next_layers_weights):
    derivatives_of_curr_activations = (curr_pre_activations > 0).astype(float) # relu derivative to Z
    delta_array = np.array(next_layers_error_gradients) @ next_layers_weights.T
    delta_array = delta_array * derivatives_of_curr_activations
    return delta_array


In [ ]:
def adam_optimizer(curr_weights, curr_gradients, iter, layer, is_bias=False):

    if is_bias:
        moment_1_curr = (beta_1*(bias_moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
        bias_moment_1_history[layer] = moment_1_curr
    else:
        moment_1_curr = (beta_1*(moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
        moment_1_history[layer] = moment_1_curr

    hat_moment_1 = moment_1_curr/(1-beta_1**iter)

    if is_bias:
        moment_2_curr = beta_2*(bias_moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
        bias_moment_2_history[layer] = moment_2_curr
    else:
        moment_2_curr = beta_2*(moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
        moment_2_history[layer] = moment_2_curr

    hat_moment_2 = moment_2_curr/(1-beta_2**iter)

    return ((learning_rate/(np.sqrt(hat_moment_2)+epsilon)) * hat_moment_1)

In [ ]:
def forward_propagation(input_layer,weights, biases):
    global activations, weighted_sums
    
    input_matrix = input_layer
    prev_activations = input_matrix
    prediction = None
    for index in range(len(weights)):
        curr_weights = weights[index]
        weighted_sum = prev_activations @ curr_weights
        bias_added_weights = weighted_sum + biases[index]
        weighted_sums.append(bias_added_weights)
        if index < len(weights)-1:
            prev_activations = np.where(bias_added_weights>0,bias_added_weights,0) # applying relu with simd
            activations.append(prev_activations)
        else:
            prediction = softmax(bias_added_weights)
            activations.append(prediction)
    return prediction
        
        





In [ ]:
def backward_propagation(input_layer, layer_weights, layer_biases, prediction_array, batch_true_labels, time_step, batch_data_size):

    global activations, weighted_sums
    
    # backprop for output layer

    output_layer_error_gradients = softmax_sparse_categorical_crossentropy_gradient(prediction_array,batch_true_labels)
    # output_layer_weights_change = activations[-2].T @ output_layer_error_gradients
    
    errors_for_backprop = {
        len(layer_weights)-1:output_layer_error_gradients
    }

    bias_changes_for_backprop = {
        len(layer_weights)-1:output_layer_error_gradients
    }

    # backprop for the rest

    for l in range(len(layer_weights)-2,-1,-1):
        delta_j_arr = hidden_layer_backprop_gradients_for_layer(weighted_sums[l],errors_for_backprop[l+1],layer_weights[l+1])
        errors_for_backprop[l] = delta_j_arr
        bias_changes_for_backprop[l] = delta_j_arr
    
    #apply adam

    for l in range(len(layer_weights)):
        if l == 0:
            prev_activations = input_layer
        else:
            prev_activations = activations[l-1]

        if l == len(layer_weights) - 1:
            delta = output_layer_error_gradients
        else:
            delta = errors_for_backprop[l]

        layer_changes = (prev_activations.T @ delta)/batch_data_size
        bias_changes = np.sum(bias_changes_for_backprop[l],axis=0,keepdims=True)/batch_data_size



        adam_optimized_changes = adam_optimizer(layer_weights[l],layer_changes,time_step,l)
        layer_weights[l][:] -= adam_optimized_changes
        adam_optimized_bias_changes = adam_optimizer(layer_biases[l],bias_changes,time_step,l,True)
        layer_biases[l][:] -= adam_optimized_bias_changes






In [ ]:
def main_loop():

    rng = np.random.default_rng()
    # Define your desired custom range [low, high)

    layer_sizes = [784, 128, 64, 32, 10]

    layer_weights = []


    layer_biases = []

    # Xavier Initialization 
    for i in range(len(layer_sizes)-1):
        n_in = layer_sizes[i]
        n_out = layer_sizes[i+1]
        limit = np.sqrt(6/(n_in+n_out))
        weights = rng.uniform(-limit,limit,size=(n_in,n_out))
        biases = np.zeros((1,n_out))
        layer_weights.append(weights)
        layer_biases.append(biases)

    epochs = 10

    global activations, weighted_sums, np_x_train_flat, int_y_train
    train_data_size = len(np_x_train_flat)
    batch_data_size = 32

    iters_per_batch = train_data_size // batch_data_size
    time_step = 1
    for i in range(epochs):
        batch_iter = 0
        for j in range(iters_per_batch):
            activations = []
            weighted_sums = []
            batch_inputs = np_x_train_flat[batch_iter:batch_iter+batch_data_size]
            batch_true_labels = int_y_train[batch_iter:batch_iter+batch_data_size]

            prediction_array = forward_propagation(batch_inputs, layer_weights, layer_biases)
            
            
            print("loss ============> ", sparse_categorical_crossentropy(prediction_array,batch_true_labels))

            accuracy(prediction_array,batch_true_labels)

            print("\n\n\n")

            backward_propagation(batch_inputs, layer_weights, layer_biases, prediction_array, batch_true_labels, time_step, batch_data_size)
            batch_iter += batch_data_size
            time_step+=1
            print("time_step: ", time_step)
        print("epoch: ", i)
    weights_dict = {f"w_{i}":w for i,w in enumerate(layer_weights)}
    bias_dict = {f"b_{i}":b for i,b in enumerate(layer_biases)}
    np.savez("nn_model.npz", **weights_dict, **bias_dict)
    


main_loop()


In [ ]:
data = np.load("nn_model.npz")
layer_sizes = [784, 128, 64, 32, 10]
loaded_weights = [np.array(data[f"w_{i}"]) for i in range(len(layer_sizes) - 1)]
loaded_biases = [np.array(data[f"b_{i}"]) for i in range(len(layer_sizes) - 1)]

In [ ]:
len(loaded_weights)

In [ ]:
test_batch_size = 32
test_samples = len(x_test_flat)
activations = []
weighted_sums = []
test_batch_iter=0
all_test_preds = []
loss_arr=[]
while test_batch_iter<test_samples:
    activations = []
    weighted_sums = []
    batch_inputs = np_x_test_flat[test_batch_iter:test_batch_iter+test_batch_size]
    batch_true_labels = int_y_test[test_batch_iter:test_batch_iter+test_batch_size]
    
    prediction_array = forward_propagation(batch_inputs, loaded_weights, loaded_biases)
    loss_arr.append(sparse_categorical_crossentropy(prediction_array,batch_true_labels))
    all_test_preds.extend(list(prediction_array))
    test_batch_iter+=32

accuracy(all_test_preds,int_y_test)
print("loss ==> ", np.mean(np.array(loss_arr)))

loss ============>  0.018970229773265134
loss ============>  0.004500640697383166
loss ============>  0.001218059445563412
loss ============>  0.229201769984437
loss ============>  0.15313128773245605
loss ============>  0.0007575728837638155
loss ============>  0.27179635200272256
loss ============>  0.23390098597666453
loss ============>  0.27361882151997646
loss ============>  0.0010907025261338165
loss ============>  0.15372495994416596
loss ============>  0.37420099519124767
loss ============>  0.0432252008230131
loss ============>  0.4037103656098094
loss ============>  0.3008105213012924
loss ============>  0.01014217459651002
loss ============>  0.005978746349580965
loss ============>  0.08818011999151401
loss ============>  0.11607562498847146
loss ============>  0.2738130167876303
loss ============>  0.24909775185500002
loss ============>  0.4761108535894964
loss ============>  0.364714464700046
loss ============>  0.04832727963788667
loss ============>  0.006642390410597502
